# Qwen3.8-Flash-Next ARC-AGI-2

Qwen3.8-Flash-Next (Qwen Community License 1.0) served on the 4 L4 GPUs by an OpenAI-compatible engine, with an ARC
solver as its client. The notebook runs offline and goes through six stages:

| stage | what |
|---|---|
| 1 inputs | find the runtime dataset, the competition data and the checkpoint and runtime (Kaggle Models, 14 parts) |
| 2 run mode | scored rerun: hidden test; save run: a short smoke test on the public evaluation split |
| 3 runtime | unpack the offline Python environment with the inference engine |
| 4 server | read the checkpoint into the page cache in parallel, start the server, wait until it answers |
| 5 solver | solve every test input within the time budget |
| 6 submission | check `submission.json`; any missing attempt becomes the input grid |

## Configuration

In [ ]:
import hashlib, json, os, re, shutil, signal, subprocess, sys, threading, time, urllib.request
from pathlib import Path

INPUT_DIR = os.environ.get("NB_INPUT_DIR", "/kaggle/input")
WORK_DIR = Path(os.environ.get("NB_WORK_DIR", "/kaggle/working"))
TMP_DIR = Path(os.environ.get("NB_TMP_DIR", "/tmp/fn"))
CKPT_MERGE_DIR = TMP_DIR / "ckpt"
# Kaggle Models (one version each): role core = checkpoint without routed experts (+ n-gram table), experts = the
# low-bit expert build, runtime = the offline Python environment archive. files: name -> exact bytes
MODELS = [
 {
  "role": "core",
  "owner": "notnitsuj",
  "slug": "qwen38-flash-next-fp8-core",
  "framework": "pyTorch",
  "parts": [
   "w-p1",
   "ple-p1",
   "ple-p2",
   "ple-p3",
   "ple-p4"
  ]
 },
 {
  "role": "experts",
  "owner": "notnitsuj",
  "slug": "qwen38-flash-next-experts",
  "framework": "pyTorch",
  "parts": [
   "k320-q34-p1",
   "k320-q34-p2",
   "k320-q34-p3",
   "k320-q34-p4"
  ]
 },
 {
  "role": "runtime",
  "owner": "notnitsuj",
  "slug": "flash-next-offline-runtime",
  "framework": "other",
  "parts": [
   "py312-cu130"
  ]
 },
 {
  "role": "experts_delta",
  "owner": "notnitsuj",
  "slug": "qwen38-flash-next-experts",
  "framework": "pyTorch",
  "parts": [
   "k320-q34-r11c-d-p1",
   "k320-q34-r11c-d-p2",
   "k320-q34-r11c-d-p3"
  ]
 },
 {
  "role": "experts_delta_mtp",
  "owner": "notnitsuj",
  "slug": "qwen38-flash-next-experts",
  "framework": "pyTorch",
  "parts": [
   "k320-q34-r1b-mtp2-p1"
  ]
 }
]
MODEL_FILES = {
 "core": {
  "LICENSE": 3235,
  "base-00001-of-00003.safetensors": 3989258848,
  "base-00002-of-00003.safetensors": 3999076624,
  "base-00003-of-00003.safetensors": 2920571808,
  "base.index.json": 140290,
  "chat_template.jinja": 8952,
  "config.json": 72514,
  "generation_config.json": 202,
  "merges.txt": 3353259,
  "ngram-00001-of-00015.safetensors": 3652591704,
  "ngram-00002-of-00015.safetensors": 3600018784,
  "ngram-00003-of-00015.safetensors": 3600018784,
  "ngram-00004-of-00015.safetensors": 3600018784,
  "ngram-00005-of-00015.safetensors": 3600018776,
  "ngram-00006-of-00015.safetensors": 3600018776,
  "ngram-00007-of-00015.safetensors": 3600018776,
  "ngram-00008-of-00015.safetensors": 3600018776,
  "ngram-00009-of-00015.safetensors": 3600018776,
  "ngram-00010-of-00015.safetensors": 3600018776,
  "ngram-00011-of-00015.safetensors": 3600018776,
  "ngram-00012-of-00015.safetensors": 3600018776,
  "ngram-00013-of-00015.safetensors": 3600018776,
  "ngram-00014-of-00015.safetensors": 3600018776,
  "ngram-00015-of-00015.safetensors": 813112202,
  "ngram.index.json": 16571,
  "preprocessor_config.json": 390,
  "tokenizer.json": 12809320,
  "tokenizer_config.json": 17928,
  "vocab.json": 6722759
 },
 "experts": {
  "LICENSE": 3235,
  "keep.json": 75412,
  "lowbit-L00.pt": 725621929,
  "lowbit-L01.pt": 725621929,
  "lowbit-L02.pt": 725621929,
  "lowbit-L03.pt": 725621929,
  "lowbit-L04.pt": 725621929,
  "lowbit-L05.pt": 725621929,
  "lowbit-L06.pt": 725621929,
  "lowbit-L07.pt": 725621929,
  "lowbit-L08.pt": 725621929,
  "lowbit-L09.pt": 725621929,
  "lowbit-L10.pt": 725621929,
  "lowbit-L11.pt": 725621929,
  "lowbit-L12.pt": 725621929,
  "lowbit-L13.pt": 725621929,
  "lowbit-L14.pt": 725621929,
  "lowbit-L15.pt": 725621929,
  "lowbit-L16.pt": 725621929,
  "lowbit-L17.pt": 725621929,
  "lowbit-L18.pt": 725621929,
  "lowbit-L19.pt": 725621929,
  "lowbit-L20.pt": 725621929,
  "lowbit-L21.pt": 725621929,
  "lowbit-L22.pt": 725621929,
  "lowbit-L23.pt": 725621929,
  "lowbit-L24.pt": 725621929,
  "lowbit-L25.pt": 725621929,
  "lowbit-L26.pt": 725621929,
  "lowbit-L27.pt": 725621929,
  "lowbit-L28.pt": 725621929,
  "lowbit-L29.pt": 725621929,
  "lowbit-L30.pt": 725621929,
  "lowbit-L31.pt": 725621929,
  "lowbit-L32.pt": 725621929,
  "lowbit-L33.pt": 725621929,
  "lowbit-L34.pt": 725621929,
  "lowbit-L35.pt": 725621929,
  "lowbit-L36.pt": 725621929,
  "lowbit-L37.pt": 725621929,
  "lowbit-L38.pt": 725621929,
  "lowbit-L39.pt": 725621929,
  "lowbit-L40.pt": 725621929,
  "lowbit-L41.pt": 725621929,
  "lowbit-L42.pt": 725621929,
  "lowbit-L43.pt": 725621929,
  "lowbit-L44.pt": 725621929,
  "lowbit-L45.pt": 725621929,
  "lowbit-L46.pt": 725621929,
  "lowbit-L47.pt": 725621929,
  "lowbit-L48.pt": 725621929,
  "router-L00.pt": 1640830,
  "router-L01.pt": 1640830,
  "router-L02.pt": 1640830,
  "router-L03.pt": 1640830,
  "router-L04.pt": 1640830,
  "router-L05.pt": 1640830,
  "router-L06.pt": 1640830,
  "router-L07.pt": 1640830,
  "router-L08.pt": 1640830,
  "router-L09.pt": 1640830,
  "router-L10.pt": 1640830,
  "router-L11.pt": 1640830,
  "router-L12.pt": 1640830,
  "router-L13.pt": 1640830,
  "router-L14.pt": 1640830,
  "router-L15.pt": 1640830,
  "router-L16.pt": 1640830,
  "router-L17.pt": 1640830,
  "router-L18.pt": 1640830,
  "router-L19.pt": 1640830,
  "router-L20.pt": 1640830,
  "router-L21.pt": 1640830,
  "router-L22.pt": 1640830,
  "router-L23.pt": 1640830,
  "router-L24.pt": 1640830,
  "router-L25.pt": 1640830,
  "router-L26.pt": 1640830,
  "router-L27.pt": 1640830,
  "router-L28.pt": 1640830,
  "router-L29.pt": 1640830,
  "router-L30.pt": 1640830,
  "router-L31.pt": 1640830,
  "router-L32.pt": 1640830,
  "router-L33.pt": 1640830,
  "router-L34.pt": 1640830,
  "router-L35.pt": 1640830,
  "router-L36.pt": 1640830,
  "router-L37.pt": 1640830,
  "router-L38.pt": 1640830,
  "router-L39.pt": 1640830,
  "router-L40.pt": 1640830,
  "router-L41.pt": 1640830,
  "router-L42.pt": 1640830,
  "router-L43.pt": 1640830,
  "router-L44.pt": 1640830,
  "router-L45.pt": 1640830,
  "router-L46.pt": 1640830,
  "router-L47.pt": 1640830,
  "router-L48.pt": 1640830
 },
 "experts_delta": {
  "LICENSE": 3235,
  "keep.json": 75412,
  "router-L00.pt": 1640830,
  "router-L01.pt": 1640830,
  "router-L02.pt": 1640830,
  "router-L03.pt": 1640830,
  "router-L04.pt": 1640830,
  "router-L05.pt": 1640830,
  "router-L06.pt": 1640830,
  "router-L07.pt": 1640830,
  "router-L08.pt": 1640830,
  "router-L09.pt": 1640830,
  "router-L10.pt": 1640830,
  "router-L11.pt": 1640830,
  "router-L12.pt": 1640830,
  "router-L13.pt": 1640830,
  "router-L14.pt": 1640830,
  "router-L15.pt": 1640830,
  "router-L16.pt": 1640830,
  "router-L17.pt": 1640830,
  "router-L18.pt": 1640830,
  "router-L19.pt": 1640830,
  "router-L20.pt": 1640830,
  "router-L21.pt": 1640830,
  "router-L22.pt": 1640830,
  "router-L23.pt": 1640830,
  "router-L24.pt": 1640830,
  "router-L25.pt": 1640830,
  "router-L26.pt": 1640830,
  "router-L27.pt": 1640830,
  "router-L28.pt": 1640830,
  "router-L29.pt": 1640830,
  "router-L30.pt": 1640830,
  "router-L31.pt": 1640830,
  "router-L32.pt": 1640830,
  "router-L33.pt": 1640830,
  "router-L34.pt": 1640830,
  "router-L35.pt": 1640830,
  "router-L36.pt": 1640830,
  "router-L37.pt": 1640830,
  "router-L38.pt": 1640830,
  "router-L39.pt": 1640830,
  "router-L40.pt": 1640830,
  "router-L41.pt": 1640830,
  "router-L42.pt": 1640830,
  "router-L43.pt": 1640830,
  "router-L44.pt": 1640830,
  "router-L45.pt": 1640830,
  "router-L46.pt": 1640830,
  "router-L47.pt": 1640830,
  "router-L48.pt": 1640830,
  "scales-L00.pt": 24754513,
  "scales-L01.pt": 24754513,
  "scales-L02.pt": 24754513,
  "scales-L03.pt": 24754513,
  "scales-L04.pt": 24754513,
  "scales-L05.pt": 24754513,
  "scales-L06.pt": 24754513,
  "scales-L07.pt": 24754513,
  "scales-L08.pt": 24754513,
  "scales-L09.pt": 24754513,
  "scales-L10.pt": 24754513,
  "scales-L11.pt": 24754513,
  "scales-L12.pt": 24754513,
  "scales-L13.pt": 24754513,
  "scales-L14.pt": 24754513,
  "scales-L15.pt": 24754513,
  "scales-L16.pt": 24754513,
  "scales-L17.pt": 24754513,
  "scales-L18.pt": 24754513,
  "scales-L19.pt": 24754513,
  "scales-L20.pt": 24754513,
  "scales-L21.pt": 24754513,
  "scales-L22.pt": 24754513,
  "scales-L23.pt": 24754513,
  "scales-L24.pt": 24754513,
  "scales-L25.pt": 24754513,
  "scales-L26.pt": 24754513,
  "scales-L27.pt": 24754513,
  "scales-L28.pt": 24754513,
  "scales-L29.pt": 24754513,
  "scales-L30.pt": 24754513,
  "scales-L31.pt": 24754513,
  "scales-L32.pt": 24754513,
  "scales-L33.pt": 24754513,
  "scales-L34.pt": 24754513,
  "scales-L35.pt": 24754513,
  "scales-L36.pt": 24754513,
  "scales-L37.pt": 24754513,
  "scales-L38.pt": 24754513,
  "scales-L39.pt": 24754513,
  "scales-L40.pt": 24754513,
  "scales-L41.pt": 24754513,
  "scales-L42.pt": 24754513,
  "scales-L43.pt": 24754513,
  "scales-L44.pt": 24754513,
  "scales-L45.pt": 24754513,
  "scales-L46.pt": 24754513,
  "scales-L47.pt": 24754513,
  "scales-L48.pt": 24754513
 },
 "experts_delta_mtp": {
  "LICENSE": 3235,
  "manifest.json": 1521,
  "mtp_ovr.safetensors": 175240488,
  "router-L48.pt": 1640830,
  "scales-L48.pt": 24754513
 },
 "runtime": {
  "LICENSE": 279,
  "fn_py312.tarbin": 9103697920,
  "fn_py312.tarbin.sha256": 65
 }
}
ROLE_DIRS = {"core": CKPT_MERGE_DIR, "experts": TMP_DIR / "experts", "runtime": TMP_DIR / "runtime_model",
             "experts_delta": TMP_DIR / "experts_delta", "experts_delta_mtp": TMP_DIR / "experts_delta_mtp"}
DELTA_ROLES = ("experts_delta", "experts_delta_mtp")    # applied in this order on top of the base build
# experts_delta present: the experts Model is the base build, linked to experts_base; the served build is rebuilt
# into TMP_DIR / "experts" from base + delta (scales and router rows) before the server starts
if any(r in MODEL_FILES for r in DELTA_ROLES):
    ROLE_DIRS["experts"] = TMP_DIR / "experts_base"
REBUILT_FILES = {'lowbit-L00.pt': 725621929, 'lowbit-L01.pt': 725621929, 'lowbit-L02.pt': 725621929, 'lowbit-L03.pt': 725621929, 'lowbit-L04.pt': 725621929, 'lowbit-L05.pt': 725621929, 'lowbit-L06.pt': 725621929, 'lowbit-L07.pt': 725621929, 'lowbit-L08.pt': 725621929, 'lowbit-L09.pt': 725621929, 'lowbit-L10.pt': 725621929, 'lowbit-L11.pt': 725621929, 'lowbit-L12.pt': 725621929, 'lowbit-L13.pt': 725621929, 'lowbit-L14.pt': 725621929, 'lowbit-L15.pt': 725621929, 'lowbit-L16.pt': 725621929, 'lowbit-L17.pt': 725621929, 'lowbit-L18.pt': 725621929, 'lowbit-L19.pt': 725621929, 'lowbit-L20.pt': 725621929, 'lowbit-L21.pt': 725621929, 'lowbit-L22.pt': 725621929, 'lowbit-L23.pt': 725621929, 'lowbit-L24.pt': 725621929, 'lowbit-L25.pt': 725621929, 'lowbit-L26.pt': 725621929, 'lowbit-L27.pt': 725621929, 'lowbit-L28.pt': 725621929, 'lowbit-L29.pt': 725621929, 'lowbit-L30.pt': 725621929, 'lowbit-L31.pt': 725621929, 'lowbit-L32.pt': 725621929, 'lowbit-L33.pt': 725621929, 'lowbit-L34.pt': 725621929, 'lowbit-L35.pt': 725621929, 'lowbit-L36.pt': 725621929, 'lowbit-L37.pt': 725621929, 'lowbit-L38.pt': 725621929, 'lowbit-L39.pt': 725621929, 'lowbit-L40.pt': 725621929, 'lowbit-L41.pt': 725621929, 'lowbit-L42.pt': 725621929, 'lowbit-L43.pt': 725621929, 'lowbit-L44.pt': 725621929, 'lowbit-L45.pt': 725621929, 'lowbit-L46.pt': 725621929, 'lowbit-L47.pt': 725621929, 'lowbit-L48.pt': 725621929, 'router-L00.pt': 1640830, 'router-L01.pt': 1640830, 'router-L02.pt': 1640830, 'router-L03.pt': 1640830, 'router-L04.pt': 1640830, 'router-L05.pt': 1640830, 'router-L06.pt': 1640830, 'router-L07.pt': 1640830, 'router-L08.pt': 1640830, 'router-L09.pt': 1640830, 'router-L10.pt': 1640830, 'router-L11.pt': 1640830, 'router-L12.pt': 1640830, 'router-L13.pt': 1640830, 'router-L14.pt': 1640830, 'router-L15.pt': 1640830, 'router-L16.pt': 1640830, 'router-L17.pt': 1640830, 'router-L18.pt': 1640830, 'router-L19.pt': 1640830, 'router-L20.pt': 1640830, 'router-L21.pt': 1640830, 'router-L22.pt': 1640830, 'router-L23.pt': 1640830, 'router-L24.pt': 1640830, 'router-L25.pt': 1640830, 'router-L26.pt': 1640830, 'router-L27.pt': 1640830, 'router-L28.pt': 1640830, 'router-L29.pt': 1640830, 'router-L30.pt': 1640830, 'router-L31.pt': 1640830, 'router-L32.pt': 1640830, 'router-L33.pt': 1640830, 'router-L34.pt': 1640830, 'router-L35.pt': 1640830, 'router-L36.pt': 1640830, 'router-L37.pt': 1640830, 'router-L38.pt': 1640830, 'router-L39.pt': 1640830, 'router-L40.pt': 1640830, 'router-L41.pt': 1640830, 'router-L42.pt': 1640830, 'router-L43.pt': 1640830, 'router-L44.pt': 1640830, 'router-L45.pt': 1640830, 'router-L46.pt': 1640830, 'router-L47.pt': 1640830, 'router-L48.pt': 1640830, 'keep.json': 75412, 'mtp_ovr.safetensors': 175240488}   # name -> bytes of the rebuilt build (None: no delta)
RUNTIME_TAR = 'fn_py312.tarbin'       # in the runtime Model: extracts to RUNTIME_PYTHON's root
RUNTIME_PYTHON = '/tmp/fn_py312/bin/python3.12'
RUNTIME_DATASET = 'notnitsuj/flash-next-arc-code'
RUNTIME_MARKER = 'fn_runtime.json'   # one file at the root of the runtime dataset
WHEEL_DIR = 'wheels'   # fallback when no runtime Model: wheels in the code dataset
PORT = 8000
SERVE_CMD = []   # {python} {ckpt} {experts} {port} {runtime} filled in at launch
SERVE_ENV = {}
SOLVER_CMD = []  # + {challenges} {out} {deadline} {port}
DEADLINE_TOTAL_S = 42150   # solver must finish by kernel start + this
SAVE_RUN_DEADLINE_S = 2400
SERVER_READY_TIMEOUT_S = 3600   # storage can be slow: be patient, restart once
PREFETCH_THREADS = 16
SUBMISSION = WORK_DIR / "submission.json"
LAUNCHER = {'script': 'kaggle_fn.sh', 'deltas_at_load': True, 'rerun': {'DT': 42150, 'KILLBY': 42600, 'SKIP48': '0', 'POLL_S': '2', 'FN_CACHE_TAR': '{code}/fn_cache.tarbin', 'KV_PACK': '1', 'FP8MM': '1', 'L4CFG': '1', 'SILU_LOCAL': '1', 'SAMPLE_SHARD': '1'}, 'save': {'DT': 2400, 'KILLBY': 2600, 'MARGIN': 240, 'ENDG': 240, 'MINCAP': 16384, 'SKIP48': '0', 'POLL_S': '2', 'FN_CACHE_TAR': '{code}/fn_cache.tarbin', 'KV_PACK': '1', 'FP8MM': '1', 'L4CFG': '1', 'SILU_LOCAL': '1', 'SAMPLE_SHARD': '1'}}   # one script runs server + solver (None = separate stages)
LAUNCHER["rerun"].update({"CONC": "48", "IMPORT_FIX": "alias", "TB_FIX": "on", "ANSWER_ORDER": "shape", "LATE_PF": "1", "DRAIN": "1", "FN_HOST_DP_FUSE": "1", "FN_HOST_LOGITS": "1", "FN_MOE_ROUTER_FIRST": "1", "FN_GDN_BA_OVERLAP": "1", "FN_MOE_HYB_FUSE": "2", "FN_MOE_CHUNK": "2176", "FN_LOAD_EC": "1", "NCCL_CUMEM_ENABLE": "0", "FN_QSA_SHARE_TOPK": "1", "FN_EP_Q8": "1", "FN_EP_Q8_RS": "1", "FN_TP_Q8_RS": "1"})
LAUNCHER["save"].update({"CONC": "48", "IMPORT_FIX": "alias", "TB_FIX": "on", "ANSWER_ORDER": "shape", "LATE_PF": "1", "DRAIN": "1", "FN_HOST_DP_FUSE": "1", "FN_HOST_LOGITS": "1", "FN_MOE_ROUTER_FIRST": "1", "FN_GDN_BA_OVERLAP": "1", "FN_MOE_HYB_FUSE": "2", "FN_MOE_CHUNK": "2176", "FN_LOAD_EC": "1", "NCCL_CUMEM_ENABLE": "0", "FN_QSA_SHARE_TOPK": "1", "FN_EP_Q8": "1", "FN_EP_Q8_RS": "1", "FN_TP_Q8_RS": "1"})
LAUNCHER["eval"] = dict(LAUNCHER["rerun"])   # save run = full-length public-eval run (scored reruns use "rerun")
LAUNCHER["eval"].update({"DT": 7200, "KILLBY": 7650, "MARGIN": 300, "ENDG": 300})


## Helpers

In [ ]:
T_KERNEL = time.time()


def log(*a):
    print(time.strftime("%H:%M:%S"), f"[{time.time() - T_KERNEL:7.0f}s]", *a, flush=True)


def walk_inputs(root):
    out = []
    for r, dns, fns in os.walk(root, followlinks=True):
        dns[:] = sorted(d for d in dns if not d.startswith("."))
        out += [os.path.join(r, f) for f in fns if not f.startswith(".")]
    return sorted(out)


def link_model(files, M):
    """link the files of every variation of Model M into its role dir; every listed file must be there with its
    exact size. The core dir also gets one model.safetensors.index.json merged from the parts' *.index.json."""
    want, dest = MODEL_FILES[M["role"]], ROLE_DIRS[M["role"]]
    pat = re.compile("/" + re.escape(M["slug"].lower()) + "/" + re.escape(M["framework"].lower()) + r"/([a-z0-9-]+)/(\d+)$")
    mounts = {}
    for d in sorted({os.path.dirname(p) for p in files}):
        mm = pat.search(d.lower())
        if mm:
            mounts.setdefault(mm.group(1), []).append(d)
    missing = [v for v in M["parts"] if v not in mounts]
    assert not missing, (f"{M['role']} parts not attached: {missing}. Attach the Kaggle Model {M['owner']}/{M['slug']}"
                         f" ({M['framework']}) variations {', '.join(M['parts'])}. Seen: {sorted(mounts) or 'none'}")
    for v in M["parts"]:
        assert len(mounts[v]) == 1, f"several versions of {M['slug']} part {v} attached: {mounts[v]}"
    by_dir = {}
    for p in files:
        by_dir.setdefault(os.path.dirname(p), {})[os.path.basename(p)] = p
    merged = {}
    for v in M["parts"]:
        for n, p in by_dir.get(mounts[v][0], {}).items():
            if n in want:
                assert n not in merged or os.path.getsize(merged[n]) == os.path.getsize(p), f"{n} differs between parts"
                merged.setdefault(n, p)
    bad = [f"missing {n}" for n in want if n not in merged]
    bad += [f"{n}: {os.path.getsize(merged[n])} bytes, expected {s}" for n, s in want.items()
            if n in merged and os.path.getsize(merged[n]) != s]
    assert not bad, f"the {M['role']} Model parts are incomplete: " + "; ".join(bad[:8])
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True)
    for n, p in merged.items():
        os.symlink(p, dest / n)
    parts_idx = sorted(n for n in merged if n.endswith(".index.json") and n != "model.safetensors.index.json")
    if M["role"] == "core" and parts_idx and "model.safetensors.index.json" not in merged:
        wmap, total = {}, 0
        for n in parts_idx:
            j = json.load(open(dest / n))
            wmap.update(j["weight_map"]); total += j.get("metadata", {}).get("total_size", 0)
        json.dump({"metadata": {"total_size": total}, "weight_map": wmap}, open(dest / "model.safetensors.index.json", "w"))
    return str(dest)


def prefetch(paths, threads=PREFETCH_THREADS, block=16 << 20, first=(), rank=None):
    """read files into the page cache with many parallel readers (one sequential reader gets far less throughput on
    the input mount); returns the thread so the server start can overlap it"""
    jobs = [(p, off) for p in paths for off in range(0, os.path.getsize(p), block)]
    jobs.sort(key=lambda j: (j[0] not in first, (rank or {}).get(j[0], 0), j[1] // (1 << 30), j[0]))      # first; rank; spread
    stat = {"bytes": 0, "t0": time.time()}
    lock = threading.Lock()

    def worker(k):
        for p, off in jobs[k::threads]:
            try:
                with open(p, "rb", buffering=0) as f:
                    f.seek(off)
                    n = len(f.read(block))
                with lock:
                    stat["bytes"] += n
            except OSError:
                pass

    ts = [threading.Thread(target=worker, args=(k,), daemon=True) for k in range(threads)]
    for t in ts:
        t.start()

    def done():
        for t in ts:
            t.join()
        dt = time.time() - stat["t0"]
        log(f"prefetch: {stat['bytes'] / 2**30:.1f} GiB in {dt:.0f} s ({stat['bytes'] / 1e6 / max(dt, 1e-9):.0f} MB/s)")

    th = threading.Thread(target=done, daemon=True)
    th.start()
    return th


def kill_tree(proc, sig=signal.SIGTERM):
    try:
        os.killpg(proc.pid, sig)
    except (ProcessLookupError, PermissionError):
        pass


def server_ready(port, timeout=5):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False


def is_grid(g):
    return isinstance(g, list) and g and all(isinstance(r, list) and r and all(isinstance(v, int) for v in r) for r in g)


def check_submission(challenges_file, path):
    """every test input gets two grids; a missing or malformed attempt becomes the input (valid ones are kept)"""
    ch = json.load(open(challenges_file))
    try:
        sub = json.load(open(path))
        sub = sub if isinstance(sub, dict) else {}
    except Exception:
        sub = {}
    out, fixed = {}, 0
    for t, v in ch.items():
        cur = sub.get(t) if isinstance(sub.get(t), list) else []
        outs = []
        for k, x in enumerate(v["test"]):
            e = cur[k] if k < len(cur) and isinstance(cur[k], dict) else {}
            a1 = e.get("attempt_1") if is_grid(e.get("attempt_1")) else None
            a2 = e.get("attempt_2") if is_grid(e.get("attempt_2")) else None
            fixed += (a1 is None) + (a2 is None)
            outs.append({"attempt_1": a1 if a1 is not None else x["input"], "attempt_2": a2 if a2 is not None else x["input"]})
        out[t] = outs
    tmp = str(path) + ".tmp"
    json.dump(out, open(tmp, "w"))
    os.replace(tmp, path)
    return {"tasks": len(out), "test inputs": sum(len(v) for v in out.values()), "attempts filled with the input": fixed}


STAGE_ERR = None    # first failing stage (later stages are skipped; the last cell still writes the submission)


def oom_shield():
    """keep the notebook kernel itself out of the host OOM killer's reach (the server tree is reset to 0)"""
    try:
        with open("/proc/self/oom_score_adj", "w") as f:
            f.write("-1000")
        log("oom shield on")
    except Exception as ex:
        log("oom shield unavailable:", repr(ex)[:120])


def emergency_submission():
    """input echo for every test input not already valid in submission.json, without any other notebook state"""
    try:
        chs = [p for p in walk_inputs(INPUT_DIR) if p.endswith("/arc-agi_test_challenges.json")]
        if chs:
            log("emergency submission:", check_submission(chs[0], SUBMISSION))
    except Exception as ex:
        log("emergency submission failed:", repr(ex)[:500])


def _monitor(path, period=120):
    def rd(p):
        try:
            return open(p).read().strip()
        except Exception:
            return None
    while True:
        try:
            mi = {l.split(":")[0]: int(l.split()[1]) >> 20 for l in open("/proc/meminfo") if l.split()[1:2]}
            cg = {k: rd(p) for k, p in (("cur", "/sys/fs/cgroup/memory.current"), ("max", "/sys/fs/cgroup/memory.max"),
                                         ("cur1", "/sys/fs/cgroup/memory/memory.usage_in_bytes"),
                                         ("max1", "/sys/fs/cgroup/memory/memory.limit_in_bytes"))}
            ev = (rd("/sys/fs/cgroup/memory.events") or "").replace("\n", ",")
            ev += ",group=" + str(rd("/sys/fs/cgroup/memory.oom.group"))
            st, top = {}, []
            for d in os.listdir("/proc"):
                if not d.isdigit():
                    continue
                try:
                    s = open(f"/proc/{d}/stat").read()
                    name, rest = s[s.index("(") + 1:s.rindex(")")], s[s.rindex(")") + 2:].split()
                    st[rest[0]] = st.get(rest[0], 0) + 1
                    top.append((int(rest[21]) * 4096 >> 20, name, d, rest[0]))
                except Exception:
                    pass
            top.sort(reverse=True)
            du = {m: shutil.disk_usage(m).free >> 30 for m in ("/tmp", "/kaggle/working") if os.path.exists(m)}
            line = (f"{time.time() - T_KERNEL:.0f}s availG={mi.get('MemAvailable')} freeG={mi.get('MemFree')} "
                    f"shmemG={mi.get('Shmem')} anonG={mi.get('AnonPages')} cachedG={mi.get('Cached')} cg={ {k: v for k, v in cg.items() if v} } "
                    f"ev=[{ev}] states={st} load={(rd('/proc/loadavg') or '')[:20]} diskfreeG={du} "
                    f"top={[(m, n, p, s) for m, n, p, s in top[:6]]}")
            with open(path, "a") as f:
                f.write(line + "\n")
        except Exception as ex:
            try:
                with open(path, "a") as f:
                    f.write(f"monitor error {ex!r}\n")
            except Exception:
                pass
        time.sleep(period)


def start_monitor():
    try:
        WORK_DIR.mkdir(parents=True, exist_ok=True)
        threading.Thread(target=_monitor, args=(str(WORK_DIR / "fn_monitor.log"),), daemon=True).start()
    except Exception as ex:
        log("monitor unavailable:", repr(ex)[:120])


oom_shield()
start_monitor()


## Stage 1: inputs

In [ ]:
if STAGE_ERR is None:
    try:
        def discover_inputs():
            files = walk_inputs(INPUT_DIR)
            rt = sorted({os.path.dirname(p) for p in files if os.path.basename(p) == RUNTIME_MARKER})
            assert len(rt) == 1, f"expected exactly one runtime dataset ({RUNTIME_DATASET}), found {rt}"
            ch = [p for p in files if p.endswith("/arc-agi_test_challenges.json")]
            ev = [p for p in files if p.endswith("/arc-agi_evaluation_challenges.json")]
            assert ch, "competition data not attached"
            inp = {"runtime": rt[0], "challenges": ch[0], "evaluation": ev[0] if ev else None,
                   "checkpoint": None, "experts": None, "runtime_model": None, "experts_delta": None,
                   "experts_delta_mtp": None}
            for M in MODELS:
                inp[{"core": "checkpoint", "experts": "experts", "runtime": "runtime_model",
                     "experts_delta": "experts_delta", "experts_delta_mtp": "experts_delta_mtp"}[M["role"]]] = link_model(files, M)
            for k, v in inp.items():
                log(f"{k:12s}", v)
            for r, fs in MODEL_FILES.items():
                log(f"{r:12s} {sum(fs.values()) / 2**30:.1f} GiB, all {len(fs)} files present with exact sizes")
            return inp


        WORK_DIR.mkdir(parents=True, exist_ok=True)
        TMP_DIR.mkdir(parents=True, exist_ok=True)
        INPUTS = discover_inputs()
        # a valid submission exists from the start (input echo), so any later failure still leaves a scoreable file
        print(check_submission(INPUTS["challenges"], SUBMISSION))
    except Exception as _ex:
        STAGE_ERR = 'inputs: ' + repr(_ex)[:2000]
        log('STAGE FAILED', STAGE_ERR)


## Stage 2: run mode
- scored rerun (`KAGGLE_IS_COMPETITION_RERUN` set): hidden test set, full budget
- save run: public evaluation split with a short deadline; `submission.json` stays the input echo

In [ ]:
if STAGE_ERR is None:
    try:
        RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
        EVAL = bool(LAUNCHER and LAUNCHER.get("eval")) and not RERUN
        if EVAL and INPUTS["evaluation"]:          # public eval x2 (240 tasks: '<id>b' copies), like the scored load
            _ch = json.load(open(INPUTS["evaluation"]))
            _so = json.load(open(INPUTS["evaluation"].replace("_challenges.json", "_solutions.json")))
            _d = WORK_DIR / "eval240"; _d.mkdir(parents=True, exist_ok=True)
            json.dump({**_ch, **{t + "b": v for t, v in _ch.items()}}, open(_d / "e240_challenges.json", "w"))
            json.dump({**_so, **{t + "b": v for t, v in _so.items()}}, open(_d / "e240_solutions.json", "w"))
            INPUTS["evaluation"] = str(_d / "e240_challenges.json")     # validation kernel: full-length public-eval run
        if RERUN:
            RUN = {"challenges": INPUTS["challenges"], "out": str(SUBMISSION), "deadline": T_KERNEL + DEADLINE_TOTAL_S}
        elif EVAL:
            assert INPUTS["evaluation"], "evaluation challenges not found in the competition data"
            RUN = {"challenges": INPUTS["evaluation"], "out": str(WORK_DIR / "eval_submission.json"),
                   "deadline": T_KERNEL + float(LAUNCHER["eval"]["DT"])}
        else:
            RUN = {"challenges": INPUTS["evaluation"] or INPUTS["challenges"], "out": str(WORK_DIR / "smoke_submission.json"),
                   "deadline": T_KERNEL + SAVE_RUN_DEADLINE_S}
        log("scored rerun" if RERUN else ("evaluation run" if EVAL else "save run"), RUN)
    except Exception as _ex:
        STAGE_ERR = 'run mode: ' + repr(_ex)[:2000]
        log('STAGE FAILED', STAGE_ERR)


## Stage 3: runtime (offline wheels)

In [ ]:
if STAGE_ERR is None:
    try:
        def install_runtime(rt):
            """the runtime Model's environment archive (preferred), else offline wheels from the code dataset"""
            if INPUTS["runtime_model"]:
                py = Path(RUNTIME_PYTHON)
                if not py.exists():
                    root = py.parents[2]                 # the archive holds <env>/bin/python3.12
                    root.mkdir(parents=True, exist_ok=True)
                    r = subprocess.run(["tar", "-C", str(root), "-xf", str(Path(INPUTS["runtime_model"]) / RUNTIME_TAR)],
                                       capture_output=True, text=True)
                    assert r.returncode == 0 and py.exists(), f"runtime archive did not unpack: {r.stderr[-2000:]}"
                log("runtime python", py)
                return str(py)
            venv = TMP_DIR / "venv"
            py = venv / "bin" / "python"
            if not py.exists():
                subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(venv)], check=True)
                wheels = Path(rt) / WHEEL_DIR
                reqs = Path(rt) / "requirements.lock"
                cmd = [str(py), "-m", "pip", "install", "--no-index", "--find-links", str(wheels), "--quiet"]
                cmd += ["-r", str(reqs)] if reqs.exists() else [str(w) for w in sorted(wheels.glob("*.whl"))]
                r = subprocess.run(cmd, capture_output=True, text=True)
                assert r.returncode == 0, r.stderr[-4000:]
            log("runtime python", py)
            return str(py)


        def env_report(py):
            """versions and host facts, logged once (the save run's environment check)"""
            for cmd in (["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                        ["ldd", "--version"],
                        [py, "-c", "import sys, torch, vllm, flashinfer; print('python', sys.version.split()[0], '| torch', "
                                   "torch.__version__, 'cuda', torch.version.cuda, '| vllm', vllm.__version__, '| flashinfer', "
                                   "flashinfer.__version__, '| gpus', torch.cuda.device_count())"]):
                try:
                    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
                    log("env:", (r.stdout or r.stderr).strip().splitlines()[0] if (r.stdout or r.stderr).strip() else f"rc {r.returncode}")
                except Exception as ex:
                    log("env:", cmd[0], "unavailable:", repr(ex)[:120])
            try:
                mem = open("/proc/meminfo").read().split("\n")[0]
                log("env:", mem, "| disk /tmp free", shutil.disk_usage("/tmp").free // 2**30, "GiB")
            except Exception:
                pass


        ORDER = [r for r in ("runtime", "experts", "experts_delta", "experts_delta_mtp", "core") if r in MODEL_FILES]     # what is needed first is read first
        PF = prefetch([str(ROLE_DIRS[r] / n) for r in ORDER for n in sorted(MODEL_FILES[r], key=lambda n: -MODEL_FILES[r][n])],
                      first={str(ROLE_DIRS["runtime"] / RUNTIME_TAR)},
                      rank={str(ROLE_DIRS[r] / n): (1 if n.startswith("ngram") else 2) if r == "core" else 3 if r.startswith("experts_delta") else 4
                            for r in ORDER if r != "runtime" for n in MODEL_FILES[r]})
        t_rt = time.time()
        PY = install_runtime(INPUTS["runtime"])
        try:                                          # the unpacked archive is never read again
            _fd = os.open(str(ROLE_DIRS["runtime"] / RUNTIME_TAR), os.O_RDONLY)
            os.posix_fadvise(_fd, 0, 0, os.POSIX_FADV_DONTNEED); os.close(_fd)
        except Exception as _ex:
            log("runtime archive cache drop skipped:", repr(_ex)[:120])
        log(f"runtime ready in {time.time() - t_rt:.0f} s")
        threading.Thread(target=env_report, args=(PY,), daemon=True).start()


        def rebuild_experts(deltas):
            """served expert build = base build with the deltas applied in order (scales, router rows, MTP-layer override;
            byte-identical to the original build); written to TMP_DIR/experts, every file checked against its expected size"""
            out, t0 = TMP_DIR / "experts", time.time()
            shutil.rmtree(out, ignore_errors=True)
            tool = Path(INPUTS["runtime"]) / "delta_build.py"
            r = subprocess.run([PY, str(tool), "build", INPUTS["experts"], *deltas, str(out)],
                               capture_output=True, text=True)
            assert r.returncode == 0, "expert rebuild failed: " + (r.stderr or r.stdout)[-3000:]
            for n, sz in (REBUILT_FILES or {}).items():
                if n == "mtp_ovr.safetensors" and len(deltas) < len(ALL_DELTAS):
                    continue                                   # fallback without the MTP-layer delta
                assert (out / n).is_file() and (out / n).stat().st_size == sz, f"rebuilt expert files missing or wrong size: {n}"
            log(f"experts rebuilt from base + {len(deltas)} delta(s) in {time.time() - t0:.0f} s: {len(list(out.iterdir()))} files")
            INPUTS["experts"] = str(out)


        ALL_DELTAS = [INPUTS[r] for r in DELTA_ROLES if INPUTS.get(r)]
        BASE_EXPERTS = INPUTS["experts"]
        INPUTS["experts_delta_at_load"] = None


        def rebuild_cascade():
            """rebuilt build with all deltas -> without the last (MTP) delta -> the base build; a save run fails loudly"""
            for k in range(len(ALL_DELTAS), 0, -1):
                try:
                    rebuild_experts(ALL_DELTAS[:k])
                    return k
                except Exception as ex:
                    assert RERUN or EVAL, repr(ex)[:3000]
                    log(f"EXPERT REBUILD FAILED with {k} delta(s):", repr(ex)[:1000])
                    shutil.rmtree(TMP_DIR / "experts", ignore_errors=True)
            INPUTS["experts"] = BASE_EXPERTS
            log("EXPERT REBUILD FAILED, serving the base build instead")
            return 0


        def preflight():
            """the server applies the deltas at load (FN_MOE_DELTA): check them first with the same code path"""
            tool = Path(INPUTS["runtime"]) / "fn_preflight.py"
            r = subprocess.run([PY, str(tool), str(PRE_CODE), BASE_EXPERTS, *ALL_DELTAS],
                               capture_output=True, text=True, timeout=900)
            ok = r.returncode == 0 and "PREFLIGHT OK" in r.stdout
            log("preflight:", (r.stdout.strip().splitlines() or ["(no output)"])[-1] if ok else "FAILED " + (r.stderr or r.stdout)[-1500:])
            return ok


        PRE_CODE = TMP_DIR / "precode"                  # the preflight needs the overlay's fnmoe.py (inside fnext.tarbin)
        if ALL_DELTAS and LAUNCHER and LAUNCHER.get("deltas_at_load"):
            PRE_CODE.mkdir(parents=True, exist_ok=True)
            for n in ("fn_preflight.py", "delta_build.py"):
                shutil.copy2(Path(INPUTS["runtime"]) / n, PRE_CODE / n)
            subprocess.run(["tar", "-C", str(PRE_CODE), "-xf", str(Path(INPUTS["runtime"]) / "fnext.tarbin")], check=True)
            ok = False
            try:
                ok = preflight()
            except Exception as ex:
                log("preflight error:", repr(ex)[:500])
            if ok:
                INPUTS["experts_delta_at_load"] = ":".join(ALL_DELTAS)
                log(f"experts: base build + {len(ALL_DELTAS)} delta(s) applied at load by the server")
            else:
                assert RERUN or EVAL, "delta preflight failed"
                log("DELTA PREFLIGHT FAILED: rebuilding the expert build instead")
                rebuild_cascade()
        elif ALL_DELTAS:
            rebuild_cascade()
    except Exception as _ex:
        STAGE_ERR = 'stage 10: ' + repr(_ex)[:2000]
        log('STAGE FAILED', STAGE_ERR)


## Stage 4-5: server and solver
The launcher script in the code dataset applies the engine patches, starts the inference server (restarted if it dies), keeps both data-parallel engines busy near the end, runs the solver against the Kaggle clock and writes the submission; any task it did not write is filled with the input.

In [ ]:
def container_start():
    """epoch of the container start (the 12 h clock anchor); the notebook start if that cannot be read"""
    try:
        ticks = int(open("/proc/1/stat").read().rsplit(")", 1)[1].split()[19])
        btime = int(next(l.split()[1] for l in open("/proc/stat") if l.startswith("btime")))
        t = btime + ticks / os.sysconf("SC_CLK_TCK")
        return t if 0 <= T_KERNEL - t < 7200 else T_KERNEL
    except Exception:
        return T_KERNEL


def run_launcher():
    t0 = container_start()
    over = LAUNCHER["rerun" if RERUN else ("eval" if EVAL else "save")]
    code_dir = TMP_DIR / "code"                  # writable copy of the code dataset + the unpacked overlay
    if not code_dir.exists():
        shutil.copytree(INPUTS["runtime"], code_dir)
        if (code_dir / "fnext.tarbin").exists():
            subprocess.run(["tar", "-C", str(code_dir), "-xf", str(code_dir / "fnext.tarbin")], check=True)
    for attempt in (1, 2):
        env = dict(os.environ, ARC_HOST_AR_BLOCKS="16", T0=str(int(t0)), PY=PY, CODE=str(code_dir), MODEL=INPUTS["checkpoint"],
                   EXPERTS=INPUTS["experts"] or "", CHAL=RUN["challenges"], OUT=RUN["out"], WORK=str(WORK_DIR / "fn"),
                   **{k: str(v).replace("{code}", str(code_dir)) for k, v in over.items()})
        if INPUTS.get("experts_delta_at_load"):
            env["EXPERTS_DELTA"] = INPUTS["experts_delta_at_load"]
        for k in [k for k in env if k.startswith(("JPY_", "PYDEVD_"))]:
            env.pop(k)
        kill_at = t0 + float(over["KILLBY"]) + 180
        log(f"launcher (attempt {attempt}):", LAUNCHER["script"], {k: env.get(k) for k in ("T0", "CHAL", "OUT", "EXPERTS", "EXPERTS_DELTA")})
        p = subprocess.Popen(["bash", "-c", 'echo 500 > /proc/self/oom_score_adj 2>/dev/null; exec bash "$0"', str(code_dir / LAUNCHER["script"])], env=env, start_new_session=True)
        while p.poll() is None and time.time() < kill_at:
            time.sleep(15)
        if p.poll() is None:
            log("launcher past its kill-by: stopping it"); kill_tree(p); time.sleep(20); kill_tree(p, signal.SIGKILL)
        log("launcher exit", p.poll())
        left = t0 + float(over["DT"]) - time.time()
        # the launcher exits 3 when the server never came up (every restart failed): relaunch once on a safer build
        if p.returncode != 3 or attempt == 2 or left < 7200:
            return p
        kill_tree(p, signal.SIGKILL)
        if INPUTS.get("experts_delta_at_load"):
            log("SERVER FAILED with deltas applied at load: relaunching on a rebuilt expert build")
            INPUTS["experts_delta_at_load"] = None
            rebuild_cascade()
        else:
            log("SERVER FAILED: relaunching on the base expert build")
            INPUTS["experts"] = BASE_EXPERTS
    return p

SERVER = None
if STAGE_ERR is None:
  try:
    SERVER = run_launcher()
  except Exception as ex:                     # never lose the submission check below
    log("launcher stage error:", repr(ex)[:500]); SERVER = None

## Stage 6: submission

In [ ]:
try:
    if globals().get("SERVER") is not None:
        kill_tree(SERVER)
    print(check_submission(INPUTS["challenges"], SUBMISSION))
    if not RERUN:
        print("save run smoke:" if not EVAL else "evaluation output:", check_submission(RUN["challenges"], RUN["out"]))
    if EVAL:
        def score_eval(chal_file, out_file):
            """ARC pass@2 per task (mean over its test inputs: 1 if either attempt equals the solution)"""
            sol_file = chal_file.replace("_challenges.json", "_solutions.json")
            if not os.path.exists(sol_file):
                log("no solutions file next to", chal_file); return None
            sols, sub = json.load(open(sol_file)), json.load(open(out_file))
            per = {}
            for t, ys in sols.items():
                att = sub.get(t, [])
                hits = [int(k < len(att) and isinstance(att[k], dict) and y in (att[k].get("attempt_1"), att[k].get("attempt_2")))
                        for k, y in enumerate(ys)]
                per[t] = sum(hits) / len(hits)
            res = {"tasks": len(per), "score": sum(per.values()), "pct": 100 * sum(per.values()) / max(1, len(per)),
                   "inputs_right": None, "per_task": per}
            json.dump(res, open(WORK_DIR / "eval_score.json", "w"), indent=1)
            return res
        r = score_eval(RUN["challenges"], RUN["out"])
        if r:
            log(f"EVAL pass@2: {r['score']:.2f} / {r['tasks']} tasks = {r['pct']:.2f}%  (eval_score.json)")
    log("done")
except Exception as _ex:
    log('submission stage error:', repr(_ex)[:1000])
finally:
    emergency_submission()
    log('stage status:', STAGE_ERR or 'all stages ran')
    try:                                  # hard teardown, after the submission is final
        if globals().get('SERVER') is not None:
            kill_tree(SERVER, signal.SIGKILL)
        for _pat in ('VLLM::', 'vllm serve'):
            subprocess.run(['pkill', '-9', '-f', _pat], timeout=30)
        log('teardown: server tree and vLLM processes killed')
    except Exception as _ex:
        log('teardown error:', repr(_ex)[:200])
# a save run must fail visibly (a scored rerun never raises: the submission above is always valid)
if STAGE_ERR and not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    raise RuntimeError(STAGE_ERR)
